In [ ]:
import sys, os
sys.path.insert(0, '/home/okonias/projects/td-mpc_o2-nf')
sys.path.insert(0, '/home/okonias/projects/td-mpc_o2-nf/tdmpc/src')

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['MUJOCO_GL'] = 'egl'

In [ ]:
import copy, time
import torch
import matplotlib.pyplot as plt
from env import make_env
from algorithm.helper import Episode, ReplayBuffer, linear_schedule
from scripts.train_o2 import make_cfg
from o2.training_utils import set_seed
from o2_base.agent import TDMPC_Dec

CKPT = '/home/okonias/projects/td-mpc_o2-nf/runs/o2_base/cheetah-run_ar1_s1/model_switch.pt'
VARIANTS = ['lml_softmax', 'softmax']    # add 'lml' for the current DCEM as a reference
WARMUP_EPISODES  = 3                     # TD-MPC episodes from the checkpoint, no updates, shared by all variants
PRETRAIN_UPDATES = 100
EPISODES         = 20                    # decoder episodes per variant
START_STEP       = 20 * 1000             # end of phase 1 at ar 1: horizon/std schedules are finished

cfg = make_cfg('cheetah-run', seed=1, action_repeat=1, horizon=5,
               dec_hidden_dim=256, dec_num_layers=2,
               latent_num_samples=32, latent_num_elites=8, iterations=5, dcem_iterations=5,
               lml_temperature=5, dcem_batch_size=64, decoder_updates=70)
cfg.episode_length   = 1000 // cfg.action_repeat
cfg.seed_steps       = 4 * cfg.episode_length
cfg.train_steps      = 40 * cfg.episode_length
cfg.horizon_schedule = f'linear(1,5,{START_STEP})'
cfg.std_schedule     = f'linear(0.5,0.05,{START_STEP})'
cfg.latent_action_dim = 100
cfg.dcem_softmax_temperature = 2.0       # TD-MPC uses 0.5; at ar 1 that is close to uniform weights
env = make_env(cfg)

In [ ]:
def load_agent(cfg):
    agent = TDMPC_Dec(cfg)
    d = torch.load(CKPT)
    agent.model.load_state_dict(d['model'])
    agent.model_target.load_state_dict(d['model_target'])
    agent.decoder.load_state_dict(d['decoder'])      # same untrained init for every variant
    agent.std = linear_schedule(cfg.std_schedule, START_STEP)
    return agent

@torch.no_grad()
def decoder_act(agent, obs, step):
    obs = torch.tensor(obs, dtype=torch.float32, device=agent.device).unsqueeze(0)
    u_mean, u_std, _ = agent.dcem(obs, step)
    u = u_mean + u_std * torch.randn_like(u_mean)
    return agent.decoder(u, agent.model.h(obs))[0, 0]

def run_episode(agent, act):
    obs = env.reset()
    episode = Episode(agent.cfg, obs)
    while not episode.done:
        action = act(obs, episode)
        obs, reward, done, _ = env.step(action.cpu().numpy())
        episode += (obs, action, reward, done)
    return episode

def decoder_updates(agent, buffer, n, step):
    rows = []
    for _ in range(n):
        obs, *_ = buffer.sample()
        rows.append(agent.update_decoder(obs[:agent.cfg.dcem_batch_size], step))
    return {k: sum(r[k] for r in rows) / n for k in rows[0]}

In [ ]:
set_seed(cfg.seed)
base = load_agent(cfg)
warmup = []
for i in range(WARMUP_EPISODES):
    ep = run_episode(base, lambda obs, e: base.plan(obs, step=START_STEP, t0=e.first))
    warmup.append(ep)
    print(f'warm-up {i}  reward {ep.cumulative_reward:.1f}')
del base

In [ ]:
results = {}
for weighting in VARIANTS:
    vcfg = copy.deepcopy(cfg)
    vcfg.dcem_weighting = weighting
    set_seed(cfg.seed)
    agent = load_agent(vcfg)
    buffer = ReplayBuffer(vcfg)
    for ep in warmup:
        buffer += ep

    pre = decoder_updates(agent, buffer, PRETRAIN_UPDATES, START_STEP)
    print(f'[{weighting}] pretrained: value {pre["value_mean"]:.2f}  eff_elites {pre["eff_elites_first"]:.1f}')

    log, start = [], time.time()
    for i in range(EPISODES):
        step = START_STEP + i * vcfg.episode_length
        episode = run_episode(agent, lambda obs, e: decoder_act(agent, obs, step))
        buffer += episode
        for j in range(vcfg.episode_length):             # TOLD keeps training, as in phase 2
            told = agent.update(buffer, step + j)
        dec = decoder_updates(agent, buffer, vcfg.decoder_updates, step)
        log.append({'reward': episode.cumulative_reward, 'value_loss': told['value_loss'], **dec})
        print(f'[{weighting}] ep {i:>2}  reward {episode.cumulative_reward:6.1f}  value {dec["value_mean"]:6.2f}  '
              f'eff_elites {dec["eff_elites_first"]:5.1f}  grad_unroll {dec["grad_unroll"]:.2e}  '
              f'time {time.time() - start:5.0f}s')
    results[weighting] = log

In [ ]:
keys = ['reward', 'eff_elites_first', 'grad_unroll']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, log in results.items():
    for ax, key in zip(axes, keys):
        ax.plot([r[key] for r in log], label=name)
for ax, key in zip(axes, keys):
    ax.set_title(key)
    ax.set_xlabel('episode')
    ax.legend()
axes[2].set_yscale('log')
plt.show()